# Per-User Profiles v2 - creative, de-templated bios (llama3.1)

v2 targets the repetition seen in `profiles_ministral_3_8b.json` (one phrase appeared 347x,
~80% of seekings opened identically). Levers: higher temperature + top_p + repeat_penalty,
per-user seed, **rotating** diverse few-shot examples (no fixed phrase in every prompt), a hard
banned-phrase list seeded from the leak, and structural rules that forbid the formulaic openers.

The deterministic structured-profile cells are reused unchanged from v1. The final **audit cell**
measures diversity vs the ministral output so the improvement is verifiable, not assumed.

Set `SAMPLE_N = 3` for a smoke test, then `None` for the full 523-user run.

In [ ]:
# --- Config ---
MODEL = "llama3.1:8b"
OLLAMA_URL = "http://localhost:11434/api/chat"
SAMPLE_N = None                    # set to None for the full 523-user run
CSV_PATH = "dataset/speed_dating_clean.csv"
MINISTRAL_PATH = "profiles_ministral_3_8b.json"   # baseline for the audit cell

# higher creativity than v1; repeat_penalty curbs within-bio repetition (seed added per user)
GEN_OPTS = {"temperature": 0.9, "top_p": 0.95, "repeat_penalty": 1.3}

import re, json
model_slug = re.sub(r"[^0-9a-zA-Z]+", "_", MODEL).strip("_")
OUT_PATH = f"profiles_v2_{model_slug}.json"
print("model:", MODEL, "| output:", OUT_PATH, "| sample:", SAMPLE_N, "| opts:", GEN_OPTS)

model: llama3.1:8b | output: profiles_v2_llama3_1_8b.json | sample: 3 | opts: {'temperature': 0.9, 'top_p': 0.95, 'repeat_penalty': 1.3}


In [2]:
# --- Load + dedupe to one row per person ---
import pandas as pd

df = pd.read_csv(CSV_PATH).drop_duplicates("iid").reset_index(drop=True)
print("unique users:", len(df))
df[["iid", "age", "gender", "field_cd", "career_c"]].head()

unique users: 523


,iid,age,gender,field_cd,career_c
0,1,21.0,0,1.0,NaN
1,2,24.0,0,1.0,NaN
2,3,25.0,0,2.0,NaN
3,4,23.0,0,1.0,1.0
4,5,21.0,0,1.0,1.0


In [3]:
# --- Decoder maps (from Speed Dating Data Key.md) ---
GENDER = {0: "Female", 1: "Male"}

FIELD = {
    1: "Law", 2: "Math", 3: "Social Science / Psychology",
    4: "Medical / Pharma / Bio Tech", 5: "Engineering",
    6: "English / Creative Writing / Journalism", 7: "History / Religion / Philosophy",
    8: "Business / Econ / Finance", 9: "Education / Academia",
    10: "Biological Sciences / Chemistry / Physics", 11: "Social Work",
    12: "Undergrad / undecided", 13: "Political Science / International Affairs",
    14: "Film", 15: "Fine Arts / Arts Administration", 16: "Languages",
    17: "Architecture", 18: "Other",
}

CAREER = {
    1: "Lawyer", 2: "Academic / Research", 3: "Psychologist", 4: "Doctor / Medicine",
    5: "Engineer", 6: "Creative Arts / Entertainment",
    7: "Banking / Consulting / Finance / Business", 8: "Real Estate",
    9: "International / Humanitarian Affairs", 10: "Undecided", 11: "Social Work",
    12: "Speech Pathology", 13: "Politics", 14: "Pro sports / Athletics",
    15: "Other", 16: "Journalism", 17: "Architecture",
}

RACE = {
    1: "Black / African American", 2: "European / Caucasian-American",
    3: "Latino / Hispanic American", 4: "Asian / Pacific Islander / Asian-American",
    5: "Native American", 6: "Other",
}

GOAL = {
    1: "a fun night out", 2: "to meet new people", 3: "to get a date",
    4: "looking for a serious relationship", 5: "to say I did it", 6: "other",
}

FREQ = {
    1: "several times a week", 2: "twice a week", 3: "once a week",
    4: "twice a month", 5: "once a month", 6: "several times a year", 7: "almost never",
}

def decode(mapping, value):
    """Look up a coded value, returning 'Unknown' for NaN / out-of-range codes."""
    if pd.isna(value):
        return "Unknown"
    return mapping.get(int(value), "Unknown")

def num(value):
    """Return a plain number (rounded) or None for NaN, for JSON-friendly output."""
    if pd.isna(value):
        return None
    return round(float(value), 2)

def sentiment(score):
    """Map a 1-10 interest score to a sentiment label (None if missing). Float-safe bins."""
    if score is None:
        return None
    if score >= 8:
        return "love it"
    if score >= 6:
        return "cool with it"
    if score >= 5:
        return "meh"
    if score >= 3:
        return "don't like it"
    return "hate it"

SENTIMENT_KEY = {  # label -> by_sentiment bucket name
    "love it": "love_it", "cool with it": "cool_with", "meh": "meh",
    "don't like it": "dislike", "hate it": "hate",
}

HOBBIES = [
    "sports", "tvsports", "exercise", "dining", "museums", "art", "hiking", "gaming",
    "clubbing", "reading", "tv", "theater", "movies", "concerts", "music", "shopping", "yoga",
]
HOBBY_LABEL = {
    "sports": "playing sports", "tvsports": "watching sports", "exercise": "exercising",
    "dining": "dining out", "museums": "museums & galleries", "art": "art",
    "hiking": "hiking & camping", "gaming": "gaming", "clubbing": "dancing & clubbing",
    "reading": "reading", "tv": "watching TV", "theater": "theater", "movies": "movies",
    "concerts": "concerts", "music": "music", "shopping": "shopping", "yoga": "yoga & meditation",
}

# 6 partner-preference traits -> (normalized weight col, self-rating col, label)
TRAITS = [
    ("attractive",       "attr1_1_norm",  "attr3_1"),
    ("sincere",          "sinc1_1_norm",  "sinc3_1"),
    ("intelligent",      "intel1_1_norm", "intel3_1"),
    ("fun",              "fun1_1_norm",   "fun3_1"),
    ("ambitious",        "amb1_1_norm",   "amb3_1"),
    ("shared_interests", "shar1_1_norm",  None),
]
print("decoders ready")

decoders ready


In [4]:
# --- Build the deterministic structured profile block ---
def build_structured_profile(row):
    # hobby interest scores (1-10) -> per-hobby sentiment + grouped-by-sentiment buckets
    scores = {h: num(row[h]) for h in HOBBIES}
    hobby_sentiment = {}
    by_sentiment = {"love_it": [], "cool_with": [], "meh": [], "dislike": [], "hate": []}
    for h in HOBBIES:
        label = sentiment(scores[h])
        if label is None:  # missing score -> skip
            continue
        hobby_sentiment[h] = label
        by_sentiment[SENTIMENT_KEY[label]].append(h)

    # partner-preference weights (normalized, cross-scale comparable) + most valued
    weights = {name: num(row[wcol]) for name, wcol, _ in TRAITS}
    rated_w = [(name, weights[name]) for name in weights if weights[name] is not None]
    most_valued = [n for n, _ in sorted(rated_w, key=lambda x: x[1], reverse=True)[:3]]

    self_perception = {name: num(row[scol]) for name, _, scol in TRAITS if scol}

    return {
        "user_id": int(row["iid"]),
        "demographics": {
            "age": num(row["age"]),
            "gender": decode(GENDER, row["gender"]),
            "race": decode(RACE, row["race"]),
            "field_of_study": decode(FIELD, row["field_cd"]),
            "career": decode(CAREER, row["career_c"]),
        },
        "dating_context": {
            "goal": decode(GOAL, row["goal"]),
            "dates_frequency": decode(FREQ, row["date"]),
            "goes_out": decode(FREQ, row["go_out"]),
            "expected_happiness": num(row["exphappy"]),
        },
        "partner_importance": {
            "same_race": num(row["imprace"]),
            "same_religion": num(row["imprelig"]),
        },
        "interests": {
            "scores": scores,
            "sentiment": hobby_sentiment,
            "by_sentiment": by_sentiment,
        },
        "partner_preferences": {"weights": weights, "most_valued": most_valued},
        "self_perception": self_perception,
    }

# quick look
import json as _json
print(_json.dumps(build_structured_profile(df.iloc[0]), indent=2))

{
  "user_id": 1,
  "demographics": {
    "age": 21.0,
    "gender": "Female",
    "race": "Asian / Pacific Islander / Asian-American",
    "field_of_study": "Law",
    "career": "Unknown"
  },
  "dating_context": {
    "goal": "to meet new people",
    "dates_frequency": "almost never",
    "goes_out": "several times a week",
    "expected_happiness": 3.0
  },
  "partner_importance": {
    "same_race": 2.0,
    "same_religion": 4.0
  },
  "interests": {
    "scores": {
      "sports": 9.0,
      "tvsports": 2.0,
      "exercise": 8.0,
      "dining": 9.0,
      "museums": 1.0,
      "art": 1.0,
      "hiking": 5.0,
      "gaming": 1.0,
      "clubbing": 5.0,
      "reading": 6.0,
      "tv": 9.0,
      "theater": 1.0,
      "movies": 10.0,
      "concerts": 10.0,
      "music": 9.0,
      "shopping": 8.0,
      "yoga": 1.0
    },
    "sentiment": {
      "sports": "love it",
      "tvsports": "hate it",
      "exercise": "love it",
      "dining": "love it",
      "museums": "hate it"

In [5]:
# --- Render a structured profile as a readable brief for the LLM ---
def _names(keys):
    return ", ".join(HOBBY_LABEL.get(h, h) for h in keys)

def profile_to_prompt(p):
    d, ctx = p["demographics"], p["dating_context"]
    bys = p["interests"]["by_sentiment"]
    age = d["age"] if d["age"] is not None else "unknown-age"
    loves = _names(bys["love_it"]) or "nothing in particular"
    enjoys = _names(bys["cool_with"])
    dislikes = _names(bys["dislike"] + bys["hate"])
    valued = ", ".join(t.replace("_", " ") for t in p["partner_preferences"]["most_valued"]) or "a good connection"
    imp = p["partner_importance"]
    lines = [
        f"Age: {age}",
        f"Gender: {d['gender']}",
        f"Race/ethnicity: {d['race']}",
        f"Field of study: {d['field_of_study']}",
        f"Career: {d['career']}",
        f"Reason for joining: {ctx['goal']}",
        f"Goes out: {ctx['goes_out']}; dates: {ctx['dates_frequency']}",
        f"Loves: {loves}",
    ]
    if enjoys:
        lines.append(f"Enjoys: {enjoys}")
    if dislikes:
        lines.append(f"Not into: {dislikes}")
    lines.append(f"Most values in a partner: {valued}")
    lines.append(
        f"Importance of same race (1-10): {imp['same_race']}; same religion (1-10): {imp['same_religion']}"
    )
    return "\n".join(lines)

print(profile_to_prompt(build_structured_profile(df.iloc[0])))

Age: 21.0
Gender: Female
Race/ethnicity: Asian / Pacific Islander / Asian-American
Field of study: Law
Career: Unknown
Reason for joining: to meet new people
Goes out: several times a week; dates: almost never
Loves: playing sports, exercising, dining out, watching TV, movies, concerts, music, shopping
Enjoys: reading
Not into: watching sports, museums & galleries, art, gaming, theater, yoga & meditation
Most values in a partner: sincere, intelligent, attractive
Importance of same race (1-10): 2.0; same religion (1-10): 4.0


In [6]:
# --- Few-shot pool (rotated per user) + banned phrases ---
# 6 structurally diverse examples: different openers, lengths, rhythms; grounded, no proper nouns.
# Shown only to illustrate RANGE. pick_examples() rotates 2 per user so no single phrase is in
# every prompt (that was the v1/ministral leak mechanism).
EXAMPLE_POOL = [
    {"bio": "Engineer who treats weekend hikes like deadlines and cooking like an experiment. Most of my best ideas turn up somewhere around mile three.",
     "seeking": "Someone curious and a little stubborn - the type who argues about where to eat and actually means it."},
    {"bio": "There's a quiet kind of happy I get when a long dinner turns into an even longer conversation. Med school eats my week, but live music always makes the cut.",
     "seeking": "Kindness first, cleverness a close second, and a soft spot for people who run five minutes late."},
    {"bio": "Three facts: I will out-dance you, lose to you at every video game, and absolutely cry at the movies. Studying psychology, partly so I can over-analyze all of it.",
     "seeking": "Bring your worst jokes and your best playlists; ambition is great, so is knowing when to put the phone down."},
    {"bio": "Most of my hours go to paint, a few to people, and never quite enough to the outdoors. Galleries feel more like a habitat than a hobby.",
     "seeking": "Thoughtful and self-aware wins every time - a real listener who's a little obsessed with something of their own."},
    {"bio": "Mornings belong to the gym, afternoons to the trail, evenings to whoever's up for the new place across town. Finance funds the whole operation.",
     "seeking": "Either match my pace or drag me off the couch - both work. Honesty over everything."},
    {"bio": "Happiest with strong coffee and a stack of half-finished novels. Teaching means I talk for a living and somehow still have more to say.",
     "seeking": "Warmth and wit in roughly equal measure, plus a person who treats slow Sundays as sacred."},
]

def pick_examples(iid, k=2):
    i = int(iid) % len(EXAMPLE_POOL)
    return [EXAMPLE_POOL[(i + j) % len(EXAMPLE_POOL)] for j in range(k)]

# seeded from the ministral leak audit (lowercased substring match)
BANNED_PHRASES = [
    "you won't catch me", "won't catch me", "i spend my days", "by night",
    "when i'm not", "you'll find me", "curled up with a book", "losing myself in a",
    "i'm looking for someone", "looking for someone who", "i'm drawn to people",
    "drawn to people who", "who can match my", "what matters most",
]

# Commonly-hallucinated brands / media the model invents at higher temp. Used only as a
# post-filter that triggers regeneration (NOT shown in the prompt, to avoid priming them).
BANNED_NAMES = [
    "netflix", "spotify", "youtube", "instagram", "tiktok", "tinder", "hinge", "bumble",
    "facebook", "twitter", "snapchat", "disney", "marvel", "hbo", "amazon", "starbucks",
    "the office", "friends", "game of thrones", "harry potter", "star wars", "taylor swift",
]
print("examples:", len(EXAMPLE_POOL), "| banned phrases:", len(BANNED_PHRASES),
      "| banned names:", len(BANNED_NAMES))

examples: 6 | banned phrases: 14 | banned names: 22


In [7]:
# --- Ollama call v2 (creative + grounded; rotated examples; per-user seed; retry-on-violation) ---
import urllib.request

SYSTEM_PROMPT = (
    "You are a sharp, versatile copywriter who writes short dating-app profiles. You receive a "
    "participant's factual brief and write two first-person paragraphs.\n\n"
    "GROUNDING (critical): use ONLY facts stated in the brief. Never invent names, place names, "
    "employers, pets, family members, numbers, or quotes. Do NOT name or reference any specific TV "
    "show, film, book, song, band, brand, app, or sports team (e.g. never write things like "
    "'quote The Office' or name a streaming service). You may add tone and phrasing, never new facts.\n\n"
    "OUTPUT: respond with ONLY a JSON object with exactly two keys - 'bio' (2-3 sentences: who "
    "they are, drawing on their field/career and what they love; you may work in ONE light dislike "
    "if a clear 'Not into' is given) and 'seeking' (1-2 sentences on what they want in a partner, "
    "centered on the traits they value; inviting, not a checklist). No markdown. Never use a name "
    "or a '[Name]' placeholder.\n\n"
    "VARIETY (critical - most of these profiles read identically; yours must not): do NOT open "
    "the bio with the job title or 'I'm a/an [job]'. Do NOT open 'seeking' with 'I'm looking for', "
    "'Looking for', or 'I'm drawn to'. Vary sentence length and rhythm. The examples show the RANGE "
    "of acceptable styles - never reuse their wording, openers, sentences, or metaphors; they are "
    "inspiration for variety, not templates.\n\n"
    "BANNED PHRASES - never use these or close variants: " + "; ".join(BANNED_PHRASES) + "."
)

def _examples_block(examples):
    lines = ["STYLE-RANGE EXAMPLES (for variety only - do NOT copy any wording):"]
    for ex in examples:
        lines.append(f"- bio: {ex['bio']}")
        lines.append(f"  seeking: {ex['seeking']}")
    return "\n".join(lines)

def sanitize(s):
    """Drop stray U+FFFD from token-boundary glitches, tidy whitespace/punctuation spacing."""
    s = s.replace("�", "")
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    return s

def _violations(text):
    """Banned stock phrases OR hallucinated brand/media names -> reject and regenerate."""
    blob = (text["bio"] + " " + text["seeking"]).lower()
    return [t for t in (BANNED_PHRASES + BANNED_NAMES) if t in blob]

def _one_call(user_content, seed, model, timeout):
    payload = {
        "model": model,
        "stream": False,
        "format": "json",
        "options": {**GEN_OPTS, "seed": int(seed)},
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
    }
    req = urllib.request.Request(
        OLLAMA_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        content = json.loads(resp.read())["message"]["content"]
    try:
        obj = json.loads(content)
    except json.JSONDecodeError:
        return {"bio": sanitize(content), "seeking": ""}
    return {"bio": sanitize(str(obj.get("bio", ""))),
            "seeking": sanitize(str(obj.get("seeking", "")))}

def call_ollama(prompt, examples, seed, model=MODEL, timeout=180, max_tries=3):
    """Generate, rejecting outputs with a banned phrase or hallucinated brand; bump seed and retry.
    Returns the first clean result, else the last attempt (residual hits show in the audit)."""
    user_content = (
        _examples_block(examples)
        + "\n\nNow write the profile for THIS person (return JSON only, in a fresh style):\n"
        + prompt
    )
    last = {"bio": "", "seeking": ""}
    for t in range(max_tries):
        out = _one_call(user_content, int(seed) + 1000 * t, model, timeout)
        if out["bio"] and not _violations(out):
            return out
        last = out
    return last

# smoke-test on user 0
_u0 = df.iloc[0]
call_ollama(profile_to_prompt(build_structured_profile(_u0)), pick_examples(_u0["iid"]), seed=int(_u0["iid"]))

{'bio': "Free time is rare for me these days - law school eats up my hours with its endless notes and cases to study, but on nights out I'm the one dancing at the top of a crowded club or cheering loudly in someone's living room.",
 'seeking': 'A genuine conversation starter would be nice. Someone who shares my love for trying new restaurants and exercising outside.'}

In [8]:
# --- Generate loop ---
rows = df if SAMPLE_N is None else df.head(SAMPLE_N)
profiles = {}
failures = []

for i, (_, row) in enumerate(rows.iterrows(), 1):
    prof = build_structured_profile(row)
    iid = prof["user_id"]
    try:
        text = call_ollama(profile_to_prompt(prof), pick_examples(iid), seed=iid)
    except Exception as e:
        text = {"bio": "", "seeking": ""}
        failures.append((iid, repr(e)))
    prof["bio"] = text["bio"]
    prof["seeking"] = text["seeking"]
    profiles[str(iid)] = prof
    if i % 10 == 0 or i == len(rows):
        print(f"  {i}/{len(rows)} done")

print("generated:", len(profiles), "| failures:", len(failures))
if failures:
    print(failures[:5])

  3/3 done
generated: 3 | failures: 0


In [9]:
# --- Write + preview ---
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(profiles, f, indent=2, ensure_ascii=False)
print("wrote", len(profiles), "profiles ->", OUT_PATH)

for k in list(profiles)[:3]:
    u = profiles[k]
    print("=" * 72)
    print("id", u["user_id"], "|", u["demographics"]["field_of_study"])
    print("BIO    :", u["bio"])
    print("SEEKING:", u["seeking"])

wrote 3 profiles -> profiles_v2_llama3_1_8b.json
id 1 | Law
BIO    : Not your typical law student - on the weekends you can find me crushing it at intramural soccer or getting my fitness fix in a studio class.
SEEKING: A genuine connection with someone who appreciates good food, live music and doesn't mind laughing at an off-key karaoke rendition every now and then.
id 2 | Law
BIO    : Downtown nights and courtroom days make for an interesting balance - I'm always on the lookout for new faces to share a cocktail with, not case law.
SEEKING: Intellect meets adventure in my ideal: someone who knows their way around art, music, or theater as much as they know how to dance till dawn.
id 3 | Math
BIO    : Numbers and patterns fill my days - I'm working to see beyond them too. Free time usually finds me on the trail or trying out a new restaurant.
SEEKING: Intelligence is everything, but being genuine isn't far behind. Sincerity in every conversation makes for an easy connection.


In [10]:
# --- Repetition audit: v2 vs ministral baseline ---
import re
from collections import Counter

def _texts(path, key):
    try:
        d = json.load(open(path, encoding="utf-8"))
    except FileNotFoundError:
        return None
    return [u[key] for u in d.values() if u.get(key)]

def opener(t, n=5):
    return " ".join(re.findall(r"[A-Za-z']+", t)[:n]).lower()

def grams(texts, n=4):
    c = Counter()
    for t in texts:
        w = re.findall(r"[A-Za-z']+", t.lower())
        for i in range(len(w) - n + 1):
            c[" ".join(w[i:i + n])] += 1
    return c

def report(name, texts):
    if not texts:
        print(f"[{name}] no data"); return
    op = Counter(opener(t) for t in texts)
    gr = grams(texts)
    print(f"[{name}]  n={len(texts)}  | top opener {op.most_common(1)[0][1]}x  "
          f"| top 4-gram {gr.most_common(1)[0][1]}x")
    print("   openers:", ", ".join(f"{p!r}:{c}" for p, c in op.most_common(5)))
    print("   4-grams:", ", ".join(f"{p!r}:{c}" for p, c in gr.most_common(5)))

for key in ("bio", "seeking"):
    print("#" * 78, "\n", key.upper())
    report("v2 " + model_slug, _texts(OUT_PATH, key))
    report("ministral", _texts(MINISTRAL_PATH, key))

# quick banned-phrase leak check on v2 output
v2 = json.load(open(OUT_PATH, encoding="utf-8"))
hits = Counter()
for u in v2.values():
    blob = (u.get("bio", "") + " " + u.get("seeking", "")).lower()
    for ph in BANNED_PHRASES:
        if ph in blob:
            hits[ph] += 1
print("\nBANNED-PHRASE HITS in v2 (want all 0):", dict(hits) or "none")

############################################################################## 
 BIO
[v2 llama3_1_8b]  n=3  | top opener 1x  | top 4-gram 1x
   openers: 'not your typical law student':1, 'downtown nights and courtroom days':1, 'numbers and patterns fill my':1
   4-grams: 'not your typical law':1, 'your typical law student':1, 'typical law student on':1, 'law student on the':1, 'student on the weekends':1
[ministral]  n=523  | top opener 63x  | top 4-gram 434x
   openers: 'i spend my days navigating':63, 'i m a lawyer by':43, 'i m an engineer by':41, 'i m a researcher by':27, 'i m a social worker':26
   4-grams: 'though you won t':434, 'you won t catch':347, 'won t catch me':347, 'you ll find me':214, 'i spend my days':181
############################################################################## 
 SEEKING
[v2 llama3_1_8b]  n=3  | top opener 1x  | top 4-gram 1x
   openers: 'a genuine connection with someone':1, 'intellect meets adventure in my':1, 'intelligence is everything but bei